In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import namedtuple, deque
import random
import matplotlib.pyplot as plt

# Dueling DQN Network
class DuelingDQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DuelingDQN, self).__init__()
        self.fc1 = nn.Linear(state_size, 128)
        self.value_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        self.advantage_stream = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )

    def forward(self, state):
        x = torch.relu(self.fc1(state))
        value = self.value_stream(x)
        advantage = self.advantage_stream(x)
        q = value + advantage - advantage.mean(dim=1, keepdim=True)
        return q

# Standard DQN Network (for DC-MA-DQN)
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )

    def forward(self, state):
        return self.net(state)

# SumTree for Prioritized Experience Replay
class SumTree:
    write = 0

    def __init__(self, capacity):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1)
        self.data = np.zeros(capacity, dtype=object)

    def _propagate(self, idx, change):
        parent = (idx - 1) // 2
        self.tree[parent] += change
        if parent != 0:
            self._propagate(parent, change)

    def _retrieve(self, idx, s):
        left = 2 * idx + 1
        right = left + 1
        if left >= len(self.tree):
            return idx
        if s <= self.tree[left]:
            return self._retrieve(left, s)
        else:
            return self._retrieve(right, s - self.tree[left])

    def total(self):
        return self.tree[0]

    def add(self, p, data):
        idx = self.write + self.capacity - 1
        self.data[self.write] = data
        self.update(idx, p)
        self.write += 1
        if self.write >= self.capacity:
            self.write = 0

    def update(self, idx, p):
        change = p - self.tree[idx]
        self.tree[idx] = p
        self._propagate(idx, change)

    def get(self, s):
        idx = self._retrieve(0, s)
        dataIdx = idx - self.capacity + 1
        return (idx, self.tree[idx], self.data[dataIdx])

# Prioritized Experience Replay Buffer
class PERReplayBuffer:
    e = 0.01
    a = 0.6
    beta = 0.4
    beta_increment = 0.001

    def __init__(self, capacity):
        self.tree = SumTree(capacity)
        self.capacity = capacity

    def _get_priority(self, error):
        return (np.abs(error) + self.e) ** self.a

    def add(self, error, sample):
        p = self._get_priority(error)
        self.tree.add(p, sample)

    def sample(self, n):
        batch = []
        idxs = []
        segment = self.tree.total() / n
        priorities = []
        self.beta = np.min([1., self.beta + self.beta_increment])
        for i in range(n):
            a = segment * i
            b = segment * (i + 1)
            s = random.uniform(a, b)
            (idx, p, data) = self.tree.get(s)
            priorities.append(p)
            batch.append(data)
            idxs.append(idx)
        sampling_probabilities = np.array(priorities) / self.tree.total()
        is_weight = np.power(self.tree.capacity * sampling_probabilities, -self.beta)
        is_weight /= is_weight.max()
        return batch, idxs, is_weight

    def update(self, idx, error):
        p = self._get_priority(error)
        self.tree.update(idx, p)

# Simplified Heterogeneous Network Environment
class HeteroNetEnv:
    def __init__(self, num_pbs=2, num_devices=10, num_subchannels=2):
        self.C = num_pbs  # Number of pico BSs
        self.K = num_devices  # Number of devices
        self.N = num_subchannels  # Number of subchannels
        self.P_max = 10  # Max power per BS
        self.Bsub = 1  # Subchannel bandwidth
        self.R_min = 1.0  # Reduced for stability
        self.power_levels = [0, 0.25, 0.5, 0.75, 1.0]  # Finer granularity
        self.action_size = len(self.power_levels) ** self.N
        self.state_size = 2 * self.N  # SINR and channel gains
        self.overlap = {0: [1, 2], 1: [0], 2: [0]}  # MBS overlaps with PBSs

    def reset(self):
        # Initialize channel gains (fixed for episode)
        self.g_m = np.random.rand(self.K, self.N) + 0.5
        self.g_p = np.random.rand(self.C, self.K, self.N) + 0.5
        self.previous_gamma = np.zeros((self.C + 1, self.K, self.N))
        # Random association, updated every 10 episodes externally
        states = []
        for j in range(self.C + 1):
            Kj = np.where(self.association == j)[0]
            if len(Kj) > 0:
                state = np.concatenate((self.previous_gamma[j, Kj[0]], self.g_p[j - 1, Kj[0]] if j > 0 else self.g_m[Kj[0]]))
            else:
                state = np.zeros(self.state_size)
            states.append(state)
        return states

    def step(self, actions):
        states = []
        rewards = []
        done = False
        p = np.zeros((self.C + 1, self.N))
        for j in range(self.C + 1):
            temp = actions[j]
            for n in range(self.N - 1, -1, -1):
                p[j, n] = self.power_levels[temp % len(self.power_levels)] * self.P_max / self.N
                temp //= len(self.power_levels)

        for j in range(self.C + 1):
            Kj = np.where(self.association == j)[0]
            if len(Kj) == 0:
                states.append(np.zeros(self.state_size))
                rewards.append(0)
                continue
            gamma = np.zeros((len(Kj), self.N))
            total_rate = 0
            total_poutage = 0
            for idx, k in enumerate(Kj):
                for n in range(self.N):
                    if j == 0:  # MBS
                        signal = p[j, n] * self.g_m[k, n]
                        interference = sum(p[c, n] * self.g_p[c - 1, k, n] for c in range(1, self.C + 1) if c in self.overlap[j])
                    else:  # PBS
                        signal = p[j, n] * self.g_p[j - 1, k, n]
                        interference = p[0, n] * self.g_m[k, n] + sum(p[c, n] * self.g_p[c - 1, k, n] for c in range(1, self.C + 1) if c != j and c in self.overlap[j])
                    gamma[idx, n] = signal / (interference + 1.0)
                rate = self.Bsub * np.sum(np.log2(1 + gamma[idx]))
                total_rate += rate
                total_poutage += 1 if rate < self.R_min else 0
            reward = total_rate / max(1, len(Kj)) - 1 * total_poutage  # λ=1, normalized by devices
            self.previous_gamma[j, Kj] = gamma
            state = np.concatenate((gamma[0] if len(Kj) > 0 else np.zeros(self.N),
                                   self.g_p[j - 1, Kj[0]] if j > 0 and len(Kj) > 0 else self.g_m[Kj[0]] if len(Kj) > 0 else np.zeros(self.N)))
            states.append(state)
            rewards.append(reward)
        return states, rewards, done, {}

# Agent
class Agent:
    def __init__(self, state_size, action_size, learning_rate, use_dueling=True):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = PERReplayBuffer(10000)
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.999  # Slower decay
        self.learning_rate = learning_rate
        self.model = DuelingDQN(state_size, action_size) if use_dueling else DQN(state_size, action_size)
        self.target_model = DuelingDQN(state_size, action_size) if use_dueling else DQN(state_size, action_size)
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.update_target_model()

    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state = torch.from_numpy(state).float().unsqueeze(0)
        self.model.eval()
        with torch.no_grad():
            action_values = self.model(state)
        self.model.train()
        return np.argmax(action_values.cpu().data.numpy())

    def replay(self, batch_size):
        samples, idxs, is_weights = self.memory.sample(batch_size)
        states, actions, rewards, next_states, dones = zip(*samples)
        states = torch.from_numpy(np.stack(states)).float()
        next_states = torch.from_numpy(np.stack(next_states)).float()
        actions = torch.LongTensor(actions).unsqueeze(1)
        rewards = torch.FloatTensor(rewards)
        dones = torch.FloatTensor(dones)
        is_weights = torch.FloatTensor(is_weights)
        q_eval = self.model(states).gather(1, actions).squeeze(1)
        q_next = self.target_model(next_states).detach()
        max_a = torch.argmax(self.model(next_states), dim=1)
        q_target = rewards + self.gamma * q_next.gather(1, max_a.unsqueeze(1)).squeeze(1) * (1 - dones)
        errors = torch.abs(q_eval - q_target).data.numpy()
        for i in range(batch_size):
            self.memory.update(idxs[i], errors[i])
        loss = (is_weights * (q_eval - q_target) ** 2).mean()
        self.optimizer.zero_grad()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
        loss.backward()
        self.optimizer.step()
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Transition tuple
Transition = namedtuple('Transition', ('state', 'action', 'reward', 'next_state', 'done'))

# Moving average for smoothing plots
def moving_average(data, window_size=10):
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

# Main training loop
def main():
    env = HeteroNetEnv(num_pbs=2, num_devices=10, num_subchannels=2)
    state_size = env.state_size
    action_size = env.action_size
    episodes = 1000
    batch_size = 32
    update_every = 4
    association_update_freq = 10  # Update association every 10 episodes

    # Learning rates for Fig. 3
    learning_rates = [0.1, 0.01, 0.001, 0.0001]
    rewards_lr = {lr: [] for lr in learning_rates}

    # Algorithms for Fig. 4
    algorithms = [('DC-MA-DDQN', True), ('DC-MA-DQN', False)]
    rewards_algo = {name: [] for name, _ in algorithms}

    # Fig. 3: Train DC-MA-DDQN with different learning rates
    for lr in learning_rates:
        print(f"\nTraining with Learning Rate: {lr}")
        agents = [Agent(state_size, action_size, lr, use_dueling=True) for _ in range(env.C + 1)]
        for e in range(episodes):
            if e % association_update_freq == 0:
                env.association = np.random.choice([0, 1, 2], size=env.K)
            states = env.reset()
            total_reward = 0
            step = 0
            for time in range(200):
                actions = [agent.act(states[j]) for j, agent in enumerate(agents)]
                next_states, rewards, done, _ = env.step(actions)
                step += 1
                total_reward += sum(rewards) / (env.C + 1)
                for j, agent in enumerate(agents):
                    error = abs(rewards[j])
                    agent.memory.add(error, Transition(states[j], actions[j], rewards[j], next_states[j], done))
                    if step > batch_size:
                        agent.replay(batch_size)
                    if step % update_every == 0:
                        agent.update_target_model()
                states = next_states
                if done:
                    break
            rewards_lr[lr].append(total_reward)
            if (e + 1) % 100 == 0:
                print(f"Learning Rate {lr}, Episode {e+1}/{episodes} - Avg Reward per Cell: {total_reward:.2f}")

    # Fig. 4: Train different algorithms with fixed learning rate (0.001)
    for algo_name, use_dueling in algorithms:
        print(f"\nTraining Algorithm: {algo_name}")
        agents = [Agent(state_size, action_size, 0.001, use_dueling) for _ in range(env.C + 1)]
        for e in range(episodes):
            if e % association_update_freq == 0:
                env.association = np.random.choice([0, 1, 2], size=env.K)
            states = env.reset()
            total_reward = 0
            step = 0
            for time in range(200):
                actions = [agent.act(states[j]) for j, agent in enumerate(agents)]
                next_states, rewards, done, _ = env.step(actions)
                step += 1
                total_reward += sum(rewards) / (env.C + 1)
                for j, agent in enumerate(agents):
                    error = abs(rewards[j])
                    agent.memory.add(error, Transition(states[j], actions[j], rewards[j], next_states[j], done))
                    if step > batch_size:
                        agent.replay(batch_size)
                    if step % update_every == 0:
                        agent.update_target_model()
                states = next_states
                if done:
                    break
            rewards_algo[algo_name].append(total_reward)
            if (e + 1) % 100 == 0:
                print(f"Algorithm {algo_name}, Episode {e+1}/{episodes} - Avg Reward per Cell: {total_reward:.2f}")

    # Plot Fig. 3: Average reward per cell under different learning rates
    plt.figure(figsize=(10, 5))
    colors = ['blue', 'orange', 'green', 'red']
    for lr, color in zip(learning_rates, colors):
        smoothed_rewards = moving_average(rewards_lr[lr])
        episodes_smoothed = range(1, len(smoothed_rewards) + 1)
        plt.plot(episodes_smoothed, smoothed_rewards, label=f'Learning Rate {lr}', color=color)
    plt.xlabel('Episode')
    plt.ylabel('Average Reward per Cell')
    plt.title('Average Reward per Cell under Different Learning Rates')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Plot Fig. 4: Algorithm convergence comparisons
    plt.figure(figsize=(10, 5))
    for (algo_name, _), color in zip(algorithms, colors[:len(algorithms)]):
        smoothed_rewards = moving_average(rewards_algo[algo_name])
        episodes_smoothed = range(1, len(smoothed_rewards) + 1)
        plt.plot(episodes_smoothed, smoothed_rewards, label=algo_name, color=color)
    plt.xlabel('Episode')
    plt.ylabel('Average Reward per Cell')
    plt.title('Algorithm Convergence Comparisons')
    plt.legend()
    plt.grid(True)
    plt.show()

if __name__ == "__main__":
    main()